![](./mira/gui/resources/pics/Logo_Sykno.svg "")

# __MiRa Data Evaluation__ - ___Jupyter Notebook___

### __Introduction__


**MiRa Data Measurement:**

- MiRa Measurements Path: `/mira_root/mira/meas/hdf5/`
- The MiRa radar data a Numpy array `radar_data_cube` with a shape of `(n_frames, n_samples_per_chirp, n_rx, n_tx, n_shape_sets)` and its `dtype` is `float32`.


**MiRa HDF5 Structure:**
- __/__
  - __Data__
    - ***Frame_Data_Cube_XXXX_ZZZZ***
      - `@delta_time`: `np.float32`
      - `@shape`: `(n_samples_per_chirp, n_rx, n_tx, n_shape_sets)`
      - `@timestamp`: `np.float32`
  - __Metadata__
    - ***mira_bgt_reg_content***
      - `@mira_bgt_reg_content`:
    - ***mira_bgt_reg_content_readable***
      - `@mira_bgt_reg_content_readable`:
    - ***mira_config***
      - `@mira_config`:
    - ***mira_radar_parameters*** 
      - `mira_radar_parameters`

***Frame_Data_Cube_XXXX_ZZZZ*** where ***XXXX*** are multiples of `4095` and ***ZZZZ*** is the frame counter with values in the range $[0, 4095]$.

- ***XXXX*** $\in \{k \cdot 4095 \mid k \in \mathbb{N}, 0 \leq k \geq \}$
- ***ZZZZ*** $\in \{0, 1, 2, \ldots, 4095\}$


**Working with HDF5:**

- List measurement structure:
```sh 
    h5ls -r mira_hdf5_filename.hdf5
```
- Inspect a specific group or dataset:
```sh
    h5dump -d /Data/Frame_Data_Cube_XXXX_ZZZZ mira_hdf5_filename.hdf5
```
- Inspect measurement meta data:
```sh
    h5dump -g /Metadata mira_hdf5_filename.hdf5
```
- Get statistics e.g. number of groups, datasets, and attributes:
```sh
    h5stat mira_hdf5_filename.hdf5
```


### __MiRa Data Evaluation__ - ___Python Code___

1. **Load python packages and MiRa HDF5 measurement file**


In [37]:
import h5py
import pickle
import numpy as np
from mira.rsys.mira_radar_sys import MIRA_RADAR_PARAMETER

file_path = './mira/meas/hdf5/MiRa6024_Measurement_08_02_2024_02_46_08.hdf5'

def print_hdf5_item_structure(file, itempath='/'):
    for item_name in file[itempath]:
        path = f'{itempath}/{item_name}' if itempath != '/' else f'/{item_name}'
        if isinstance(file[itempath + '/' + item_name], h5py.Group):
            print_hdf5_item_structure(file, path)


2. **Open MiRa HDF5 measurement file, load meta data and build radar data cube**

In [38]:
with h5py.File(file_path, 'r') as file:
    print_hdf5_item_structure(file)
    data_group = file['Data']

    datasets = []
    for dataset_name in file['/Data']:
        data = np.array(file['/Data'][dataset_name][:], dtype=np.float32)
        datasets.append(np.expand_dims(data, axis=0))

    data_cube = np.concatenate(datasets, axis=0, dtype=np.float32) 
    
    # Reading a specific dataset
    frame_data_cube = data_group['Frame_Data_Cube_0000_0001']
    delta_time = frame_data_cube.attrs['delta_time']
    shape = frame_data_cube.attrs['shape']
    timestamp = frame_data_cube.attrs['timestamp']
    frame_data_cube = np.array(frame_data_cube, dtype=np.float32)

    # Accessing the 'Metadata' group
    metadata_group = file['Metadata']

    # Reading 'mira_config'
    mira_config = metadata_group['mira_config'].attrs['mira_config']

    # Reading 'mira_bgt_reg_content'
    mira_bgt_reg_content = metadata_group['mira_bgt_reg_content'].attrs['mira_bgt_reg_content']

    # Reading 'mira_bgt_reg_content_readable'
    mira_bgt_reg_content_readable = metadata_group['mira_bgt_reg_content_readable'].attrs['mira_bgt_reg_content_readable']

    # Example: Reading content from 'mira_radar_parameters'
    mira_radar_parameters = metadata_group['mira_radar_parameters']['mira_radar_parameters']

    # Retrieve the numpy.void object and convert it back to bytes
    radar_param_data = mira_radar_parameters[()].tobytes()
    
    # Unpickle the bytes to get back the original Python object
    radar_param: MIRA_RADAR_PARAMETER = pickle.loads(radar_param_data)
    


3. **MiRa HDF5 measurement - data profiling**

    1. __Check MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes
        - __sys__: sub-class contains __Radar__ specific values
        - __dsp__: sub-class contains __DSP__ parameters
        - __mon__: sub-class to __Monitor__ frame count, duration time, etc.
        - __gui__: sub-class contains __GUI__ relevant parameters
        - __meas__: sub-class contains __Measurement__ relevant parameters
        - __remt__: sub-class contains __Remote Control__ relevant parameters
        - __rply__: sub-class contains __Replay Control__ relevant parameters 

In [39]:
print(f"{radar_param=}")
print(f"{radar_param.dsp=}")
print(f"{radar_param.gui=}")
print(f"{radar_param.mon=}")
print(f"{radar_param.sys=}")
print(f"{radar_param.meas=}")
print(f"{radar_param.remt=}")
print(f"{radar_param.rply=}")

radar_param=<mira.rsys.mira_radar_sys.MIRA_RADAR_PARAMETER object at 0x7fcd02431d80>
radar_param.dsp=<mira.rsys.mira_radar_sys.MIRA_RADAR_DSP_PARAMETER object at 0x7fcd02431cc0>
radar_param.gui=<mira.rsys.mira_radar_sys.MIRA_RADAR_GUI_PARAMETER object at 0x7fcd024337c0>
radar_param.mon=<mira.rsys.mira_radar_sys.MIRA_RADAR_MON_PARAMETER object at 0x7fcd02432380>
radar_param.sys=<mira.rsys.mira_radar_sys.MIRA_RADAR_SYS_PARAMETER object at 0x7fcd024303d0>
radar_param.meas=<mira.rsys.mira_radar_sys.MIRA_RADAR_MEAS_PARAMETER object at 0x7fcd02431840>
radar_param.remt=<mira.rsys.mira_radar_sys.MIRA_RADAR_REMT_PARAMETER object at 0x7fcd02433a60>
radar_param.rply=<mira.rsys.mira_radar_sys.MIRA_RADAR_RPLY_PARAMETER object at 0x7fcd02430970>


3. **MiRa HDF5 measurement - data profiling**

    2. __Check MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes

In [40]:
print(f"{data_cube.shape=}")
print(f"{frame_data_cube.shape=}\n{delta_time=}\n{shape=}\n{timestamp=}")

for shape in range(16):
    print(f"{frame_data_cube[:5,:,:,shape]}")

# print(f"{mira_bgt_reg_content=}")
# print(f"{mira_bgt_reg_content_readable=}")
print(f"{mira_radar_parameters=}")
file.close()

data_cube.shape=(112, 512, 4, 2, 16)
frame_data_cube.shape=(512, 4, 2, 16)
delta_time=0.11606287956237793
shape='(512, 4, 2, 16)'
timestamp=1707356768.7068672
[[[2054. 2054.]
  [2040. 2041.]
  [2050. 2050.]
  [2058. 2048.]]

 [[2044. 2045.]
  [2041. 2019.]
  [2044. 2040.]
  [2076. 2064.]]

 [[2038. 2042.]
  [2031. 2030.]
  [2022. 2038.]
  [2074. 2048.]]

 [[2020. 2030.]
  [2012. 2038.]
  [1976. 2035.]
  [2080. 2020.]]

 [[1986. 2008.]
  [2000. 2026.]
  [1905. 2028.]
  [2108. 1999.]]]
[[[2054. 2054.]
  [2040. 2040.]
  [2051. 2050.]
  [2056. 2048.]]

 [[2038. 2042.]
  [2038. 2020.]
  [2036. 2042.]
  [2080. 2064.]]

 [[2026. 2038.]
  [2025. 2032.]
  [1998. 2040.]
  [2082. 2046.]]

 [[2002. 2024.]
  [2005. 2036.]
  [1952. 2036.]
  [2088. 2018.]]

 [[1976. 1990.]
  [1994. 2020.]
  [1896. 2016.]
  [2110. 1994.]]]
[[[2054. 2052.]
  [2040. 2041.]
  [2050. 2048.]
  [2057. 2048.]]

 [[2042. 2041.]
  [2042. 2020.]
  [2046. 2042.]
  [2074. 2064.]]

 [[2038. 2036.]
  [2032. 2034.]
  [2024. 2038.]
 